In [1]:
%pip install --upgrade pip
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
%pip install --upgrade transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = Path("smollm2_model")

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Downloading model...")

%pip install torch --index-url https://download.pytorch.org/whl/cpu

print("Saving model locally...")

tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR, safe_serialization=True)

print("Model downloaded successfully!")
print("Model location:", MODEL_DIR.resolve())

Note: you may need to restart the kernel to use updated packages.


c:\Users\hp\AppData\Local\Python\pythoncore-3.10-64\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Ensure dependencies are available in this notebook kernel
%pip install --upgrade pip
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)

MODEL_DIR = "smollm2_model"

print("Loading local model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model.to("cpu")
model.eval()

print("Model loaded successfully!")
print("Running on CPU.")

     ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
      --------------------------------------- 0.0/1.8 MB 1.9 MB/s eta 0:00:01
     -- ------------------------------------- 0.1/1.8 MB 1.3 MB/s eta 0:00:02
     ------ --------------------------------- 0.3/1.8 MB 2.4 MB/s eta 0:00:01
     ------------ --------------------------- 0.6/1.8 MB 3.5 MB/s eta 0:00:01
     ---------------------- ----------------- 1.0/1.8 MB 5.4 MB/s eta 0:00:01
     ------------------------------------ --- 1.7/1.8 MB 7.1 MB/s eta 0:00:01
     ------------------------------------ --- 1.7/1.8 MB 7.1 MB/s eta 0:00:01
     ------------------------------------ --- 1.7/1.8 MB 7.1 MB/s eta 0:00:01
     ---------------------------------------  1.8/1.8 MB 5.2 MB/s eta 0:00:01
     ---------------------------------------- 1.8/1.8 MB 4.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
Note: 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.8 MB 13.9 MB/s eta 0:00:01
   -------- ------------------------------- 2.6/12.8 MB 7.2 MB/s eta 0:00:02
   -------- ------------------------------- 2.9/12.8 MB 6.2 MB/s eta 0:00:02
   ------------- -------------------------- 4.5/12.8 MB 5.5 MB/s eta 0:00:02
   ----------------- ---------------------- 5.8/12.8 MB 6.1 MB/s eta 0:00:02
   -------------------- ------------------- 6.6/12.8 MB 6.1 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.8 MB 5.1 MB/s eta 0:00:02
   ----------------------- ---------------- 7.6/12.8 MB 4.6 MB/s eta 0:00:02
   ------------------------- -------------- 8.1/12.8 MB 4.5 MB/s eta 0:00:02
   --------------------------- ------------ 8.9/12.8 MB 4.3 MB/s eta 0:00:01
   ----------------------------- ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
c:\Users\hp\AppData\Local\Python\pythoncore-3.10-64\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
SYSTEM_PROMPT = """
You are a helpful AI teaching assistant.
Explain concepts in simple English.
Use examples whenever helpful.
If you are unsure about an answer, say so.
"""

MAX_NEW_TOKENS = 120
MAX_HISTORY_MESSAGES = 6


def generate_response(user_message, history=None):

    if history is None:
        history = []

    # Keep recent conversation messages
    recent_history = history[-MAX_HISTORY_MESSAGES:]

    # Prepare the conversation
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ] + recent_history + [
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Convert conversation into model input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Generate a response
    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output_ids[0][input_length:]

    # Convert tokens to readable text
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        answer = "I could not generate a response. Please try again."

    return answer


print("Chatbot function created successfully!")

Chatbot function created successfully!


In [ ]:
question = "Explain Generative AI in simple terms."

answer = generate_response(question)

print("You:", question)
print("\nGenAI Chatbot:", answer)

You: Explain Generative AI in simple terms.

GenAI Chatbot: Generative AI is a type of artificial intelligence that creates new data or content from existing ones. It uses algorithms and mathematical formulas to generate something entirely different from what was given as input. This can be anything from creating art, music, or even writing stories, but it can also be used for generating text like sentences or paragraphs.

Think of it like this: imagine you have a recipe for making chocolate cake. A generative AI system could take that recipe and use its own algorithms to create a new recipe for making chocolate cakes. The ingredients and steps would still be the same, but the


In [ ]:
chat_history = []

question = "What is Machine Learning?"

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: What is Machine Learning?

Bot: Machine learning is like teaching computers to learn and make decisions on their own, without being told what to do every single time. It's kind of like how you learned to ride a bike - you had someone teach you the first few steps, but eventually, you could balance and pedal all by yourself! In computer science, this process involves giving machines data and asking them to figure out patterns or rules based on those data points. The more data they receive, the better they get at making good decisions. This way, machines can become smarter over time, just like you got wiser with practice.


In [ ]:
question = "Give me a real-life example of it."

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: Give me a real-life example of it.

Bot: A great example of machine learning is self-driving cars. Imagine if your car didn't have a human driver sitting behind the wheel. To drive safely, it needs to be able to detect obstacles, road conditions, and pedestrians. 

A company called Waymo uses machine learning to teach its cars to recognize these things through lots of training data. They show the cars pictures of different roads, people, and other objects. Then, when the car drives for the first time, it starts learning from that data. Over time, it becomes faster and more accurate at driving itself. 

For instance


In [ ]:
def start_chat():

    history = []

    print("=" * 50)
    print("       OFFLINE GENAI CHATBOT")
    print("=" * 50)
    print("Ask any question.")
    print("Type /reset to clear history.")
    print("Type /exit to stop.")
    print("=" * 50)

    while True:

        question = input("\nYou: ").strip()

        if not question:
            continue

        if question.lower() == "/exit":
            print("Bot: Goodbye!")
            break

        if question.lower() == "/reset":
            history.clear()
            print("Bot: Conversation history cleared.")
            continue

        print("Bot: Thinking...")

        answer = generate_response(question, history)

        print("\nBot:", answer)

        history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ])

        # Keep the history bounded
        if len(history) > 12:
            history = history[-12:]


# Run the chatbot
start_chat()
def start_chat():
        history = []

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            if len(history) > 12:
                history = history[-12:]

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            history = history[-12:]


start_chat()
def start_chat():
        history = []

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            if len(history) > 12:
                history = history[-12:]

start_chat()
def start_chat():
        history = []

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            history = history[-12:]


def start_chat():
        history = []

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            history = history[-12:]


def start_chat():
        history = []

        print("=" * 50)
        print("       OFFLINE GENAI CHATBOT")
        print("=" * 50)
        print("Ask any question.")
        print("Type /reset to clear history.")
        print("Type /exit to stop.")
        print("=" * 50)

        while True:
            question = input("\nYou: ").strip()

            if not question:
                continue

            if question.lower() == "/exit":
                print("Bot: Goodbye!")
                break

            if question.lower() == "/reset":
                history.clear()
                print("Bot: Conversation history cleared.")
                continue

            print("Bot: Thinking...")
            answer = generate_response(question, history)
            print("\nBot:", answer)

            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ])

            # Keep the history bounded
            history = history[-12:]


start_chat()

       OFFLINE GENAI CHATBOT
Ask any question.
Type /reset to clear history.
Type /exit to stop.
